# Working with Geospatial Data

This guide shows you how to:

- Store and query **geometry data** (points, polygons, lines) in Pixeltable
- Use built-in **geometry UDFs** for spatial computations (area, distance, intersections)
- Import geospatial data from **GeoDataFrames** and **shapefiles**
- **Visualize** geometries on interactive maps directly in your notebook

Pixeltable stores geometry as GeoJSON (JSONB) and uses [Shapely](https://shapely.readthedocs.io/) for spatial operations.

In [ ]:
%pip install -qU pixeltable

In [ ]:
import pixeltable as pxt
from shapely.geometry import Point, Polygon

pxt.drop_dir('geo_tutorial', force=True)
pxt.create_dir('geo_tutorial')

## Creating a Geometry Table

Use `pxt.Geometry` as the column type. You can insert either GeoJSON dicts or Shapely objects.

In [ ]:
t = pxt.create_table(
    'geo_tutorial/places', {'name': pxt.String, 'location': pxt.Geometry}
)

t.insert(
    [
        {'name': 'Eiffel Tower', 'location': Point(2.2945, 48.8584)},
        {
            'name': 'Statue of Liberty',
            'location': Point(-74.0445, 40.6892),
        },
        {
            'name': 'Sydney Opera House',
            'location': Point(151.2153, -33.8568),
        },
        {'name': 'Big Ben', 'location': Point(-0.1246, 51.5007)},
        {'name': 'Colosseum', 'location': Point(12.4924, 41.8902)},
    ]
)

In [ ]:
t.collect()

Geometry columns render as inline maps when the result set is small (up to 20 rows). For larger results, they fall back to JSON text.

## Geometry UDFs

Pixeltable provides geometry methods and properties that work as computed columns. They follow the Shapely API.

### Working with Polygons

In [ ]:
parks = pxt.create_table(
    'geo_tutorial/parks', {'name': pxt.String, 'boundary': pxt.Geometry}
)

# Central Park (simplified bounding box)
central_park = Polygon(
    [
        (-73.9812, 40.7681),
        (-73.9580, 40.8006),
        (-73.9494, 40.7968),
        (-73.9730, 40.7644),
        (-73.9812, 40.7681),
    ]
)

# Hyde Park (simplified bounding box)
hyde_park = Polygon(
    [
        (-0.1870, 51.5075),
        (-0.1526, 51.5136),
        (-0.1506, 51.5050),
        (-0.1850, 51.4990),
        (-0.1870, 51.5075),
    ]
)

parks.insert(
    [
        {'name': 'Central Park', 'boundary': central_park},
        {'name': 'Hyde Park', 'boundary': hyde_park},
    ]
)

### Properties: area, length, geom_type

In [ ]:
parks.add_computed_column(area=parks.boundary.area)
parks.add_computed_column(perimeter=parks.boundary.length)
parks.add_computed_column(gtype=parks.boundary.geom_type)
parks.add_computed_column(valid=parks.boundary.is_valid)

parks.select(
    parks.name, parks.area, parks.perimeter, parks.gtype, parks.valid
).collect()

### Methods: buffer, centroid, convex_hull, simplify, envelope

In [ ]:
parks.add_computed_column(center=parks.boundary.centroid())
parks.add_computed_column(hull=parks.boundary.convex_hull())
parks.add_computed_column(env=parks.boundary.envelope())

parks.select(parks.name, parks.center, parks.hull, parks.env).collect()

### Binary Methods: distance, intersection, contains

In [ ]:
pairs = pxt.create_table(
    'geo_tutorial/pairs', {'a': pxt.Geometry, 'b': pxt.Geometry}
)

poly_a = Polygon([(0, 0), (10, 0), (10, 10), (0, 10), (0, 0)])
poly_b = Polygon([(5, 5), (15, 5), (15, 15), (5, 15), (5, 5)])

pairs.insert([{'a': poly_a, 'b': poly_b}])

pairs.add_computed_column(dist=pairs.a.distance(pairs.b))
pairs.add_computed_column(inter=pairs.a.intersection(pairs.b))
pairs.add_computed_column(does_intersect=pairs.a.intersects(pairs.b))
pairs.add_computed_column(a_contains_b=pairs.a.contains(pairs.b))

pairs.select(
    pairs.dist, pairs.inter, pairs.does_intersect, pairs.a_contains_b
).collect()

### Conversion: to_wkt and from_wkt

In [ ]:
import pixeltable.functions.geometry as geo

wkt_table = pxt.create_table('geo_tutorial/wkt', {'wkt_str': pxt.String})
wkt_table.insert([{'wkt_str': 'POLYGON ((0 0, 5 0, 5 5, 0 5, 0 0))'}])
wkt_table.add_computed_column(geom=geo.from_wkt(wkt_table.wkt_str))

wkt_table.collect()

## Importing from GeoDataFrames

Pixeltable can import directly from a `geopandas.GeoDataFrame`. Geometry columns are automatically detected.

In [ ]:
import geopandas

gdf = geopandas.GeoDataFrame(
    {
        'city': ['Tokyo', 'Delhi', 'Shanghai', 'São Paulo', 'Mumbai'],
        'population_m': [37.4, 32.9, 29.2, 22.0, 21.7],
        'geometry': [
            Point(139.6917, 35.6895),
            Point(77.1025, 28.7041),
            Point(121.4737, 31.2304),
            Point(-46.6333, -23.5505),
            Point(72.8777, 19.0760),
        ],
    }
)

cities = pxt.io.import_geodataframe('geo_tutorial/cities', gdf)
cities.collect()

### Exporting back to GeoDataFrame

In [ ]:
result_gdf = cities.collect().to_geodataframe()
result_gdf

## Visualizing Geometry on a Map

`pxt.show_map()` renders all geometries from a table column on an interactive [Folium](https://python-visualization.github.io/folium/) map. You can pass `tooltip_cols` to display additional column data on hover.

In [ ]:
pxt.show_map(cities, 'geometry', tooltip_cols=['city', 'population_m'])

In [ ]:
pxt.show_map(parks, 'boundary', tooltip_cols=['name'])

### Limiting rows

In [ ]:
pxt.show_map(t, 'location', tooltip_cols=['name'], limit=3)

### Grouping geometries on one map

You can also call `.show_map()` directly on a collected result set. This is useful when you want to filter rows first and display them together on a single map. Non-geometry columns are automatically included as tooltips.

In [ ]:
t.where(t.name.isin(['Eiffel Tower', 'Statue of Liberty'])).select(
    t.name, t.location
).collect().show_map()

## Cleanup

In [ ]:
pxt.drop_dir('geo_tutorial', force=True)

## Next Steps

- Import shapefiles and GeoJSON files with `pxt.io.import_geofile()`
- Use `pxt.Geometry['POINT']` to constrain a column to a specific geometry type
- Combine geometry UDFs with other Pixeltable features like computed columns and views